# Task 24 - Feature Importance (Trees, SHAP) and Recursive Feature Elimination

**Goal:** analyse feature importance with decision trees / random forests, **compare rankings across models**, visualise **SHAP** values, and use **RFE** to find a smaller feature set that keeps accuracy while cutting model cost.

**Target (`TARGET_MODE`):**
- `"sales"` (default): above-median `units_sold`
- `"rating_positive"`: positive class from Task 23's `rating_sentiment`

**Leakage control (important):** the cumulative dataset contains columns built from the target. They are excluded automatically: for `sales` that is `units_sold`, `rating_count` and rating vote counts, plus the engagement, channel-interaction (`ch_*`), lifecycle (`lc_*`), target-encoded (`te_*`) columns and every column built from the *shrunk rating* (`rs_*`, `rating_sentiment*`, `rating_confident`), because shrinkage depends on `rating_count`, which tracks sales; for `rating_positive` it is `rating`, `rs_*`, `rating_sentiment*`, sentiment-model columns and engagement. The printed exclusion list shows exactly what was removed.

**Models compared:** decision tree, random forest, extra trees, gradient boosting (impurity importance), permutation importance (random forest and gradient boosting, on the hold-out split) and mean |SHAP| (random forest and gradient boosting).
**RFE:** `RFECV` with a random forest on the **training split only**; the chosen subset is judged on the untouched hold-out split.

**Input:** newest file in `data/processed/`. **Outputs:** `data/processed/womens_clothing_feature_selected.csv` (id, target, selected features), `reports/task_24_feature_importance_comparison.csv`, `reports/task_24_rfe_results.csv`, `reports/task_24_model_efficiency.csv`, `reports/task_24_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

# Tasks 22-24 outputs sit in front of everything.
LATE2 = ["womens_clothing_rating_sentiment_encoded.csv", "womens_clothing_pca_features.csv"]
ALL = LATE2 + FULL

import time, pickle
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
TARGET_MODE = "sales"          # "sales" or "rating_positive"
TOP_N = 15
RFE_STEP = 0.05                # fraction of features removed per RFE round

## 1. Load data, define the target and the leakage-safe feature list

In [ ]:
df, src = load_first(*[PROC / n for n in ALL])
units_col = pick(df, "units_sold"); id_col = pick(df, "product_id", required=False)
if TARGET_MODE == "sales":
    y = (df[units_col] > df[units_col].median()).astype(int); target_name = "high_sales"
    excl_exact = {units_col, "rating_count", "rating_one_count", "rating_two_count", "rating_three_count", "rating_four_count", "rating_five_count", "inventory_total"}
    excl_prefix = ("eng_", "engagement", "is_highly", "ch_", "lc_", "lifecycle", "te_", "price_tier_tree", "price_tier_dynamic", "rating_sentiment", "rs_", "rating_confident", "ttl_rating_reliable")
else:
    pos = pick(df, "rating_sentiment_pos_vs_rest"); y = df[pos].astype(int); target_name = "rating_positive"
    excl_exact = {pick(df, "rating"), "rating_count"}
    excl_prefix = ("rs_", "rating_sentiment", "rating_confident", "ttl_rating_reliable", "eng_", "engagement", "is_highly", "sentiment_", "dl_sentiment", "tfidf_sentiment_prob", "te_", "price_tier_tree", "price_tier_dynamic", "lc_", "lifecycle", "ch_")
num = df.select_dtypes(include=["number", "bool"]).copy()
num = num.astype({c: int for c in num.columns if num[c].dtype == bool})
drop = [c for c in num.columns if c in excl_exact or c.lower().startswith(excl_prefix) or c.lower().startswith("unnamed") or (id_col and c == id_col) or re.search(r"(^|_)id$", c.lower())]
X = num.drop(columns=drop)
X = X.loc[:, X.notna().any() & (X.nunique() > 1)]
X = X.fillna(X.median())
print(f"Target: {target_name} | positive share {y.mean():.2f} | rows {len(X):,} | candidate features {X.shape[1]}")
print("Excluded (leakage / ids):", sorted(drop))

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
models = {"DecisionTree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=RANDOM_STATE),
          "RandomForest": RandomForestClassifier(300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
          "ExtraTrees": ExtraTreesClassifier(300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
          "GradientBoosting": GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_STATE)}
perf = {}
for n, m in models.items():
    m.fit(Xtr, ytr); perf[n] = {"train_auc": roc_auc_score(ytr, m.predict_proba(Xtr)[:, 1]), "holdout_auc": roc_auc_score(yte, m.predict_proba(Xte)[:, 1])}
pd.DataFrame(perf).T.round(4)

## 2. Importance from several models

In [ ]:
imp = pd.DataFrame(index=X.columns)
for n, m in models.items(): imp[f"impurity_{n}"] = m.feature_importances_
for n in ["RandomForest", "GradientBoosting"]:
    pi = permutation_importance(models[n], Xte, yte, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    imp[f"permutation_{n}"] = np.clip(pi.importances_mean, 0, None)
imp.sort_values("impurity_RandomForest", ascending=False).head(TOP_N).round(4)

## 3. SHAP values

In [ ]:
def shap_pos(model, A):
    sv = shap.TreeExplainer(model)(A).values
    return sv[:, :, -1] if sv.ndim == 3 else sv
sv_gb = shap_pos(models["GradientBoosting"], Xte); sv_rf = shap_pos(models["RandomForest"], Xte)
imp["shap_GradientBoosting"] = np.abs(sv_gb).mean(axis=0); imp["shap_RandomForest"] = np.abs(sv_rf).mean(axis=0)
plt.figure(); shap.summary_plot(sv_gb, Xte, max_display=TOP_N, show=False); plt.title("SHAP summary - gradient boosting (hold-out)"); plt.tight_layout(); plt.show()
plt.figure(); shap.summary_plot(sv_rf, Xte, plot_type="bar", max_display=TOP_N, show=False); plt.title("Mean |SHAP| - random forest (hold-out)"); plt.tight_layout(); plt.show()

In [ ]:
top3 = imp["shap_GradientBoosting"].sort_values(ascending=False).index[:3]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, f in zip(ax, top3):
    shap.dependence_plot(f, sv_gb, Xte, interaction_index=None, ax=a, show=False); a.set_title(f"SHAP dependence: {f}")
plt.tight_layout(); plt.show()

## 4. Compare rankings across models

In [ ]:
imp_n = imp / imp.sum()                                                    # each method sums to 1 so scales are comparable
ranks = imp.rank(ascending=False, method="average")
rho = ranks.corr(method="spearman")
plt.figure(figsize=(9, 7)); sns.heatmap(rho, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1); plt.title("Spearman correlation of feature rankings"); plt.show()
top_sets = {c: set(imp[c].sort_values(ascending=False).index[:TOP_N]) for c in imp.columns}
jac = pd.DataFrame({a: {b: len(top_sets[a] & top_sets[b]) / len(top_sets[a] | top_sets[b]) for b in imp.columns} for a in imp.columns})
plt.figure(figsize=(9, 7)); sns.heatmap(jac, annot=True, fmt=".2f", cmap="Blues"); plt.title(f"Top-{TOP_N} overlap (Jaccard)"); plt.show()
cons = pd.DataFrame({"mean_rank": ranks.mean(axis=1), "rank_std": ranks.std(axis=1), "mean_normalised_importance": imp_n.mean(axis=1),
                     "n_methods_in_top": pd.Series({f: sum(f in s for s in top_sets.values()) for f in imp.index})}).sort_values("mean_rank")
cons.head(TOP_N).round(4)

In [ ]:
t = imp_n.loc[cons.index[:TOP_N]]
t.plot.barh(stacked=True, figsize=(10, 7), colormap="tab20"); plt.gca().invert_yaxis(); plt.title("Top features by consensus rank (stacked normalised importance)"); plt.legend(fontsize=7, bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()
print("Average pairwise rank correlation between methods:", round(rho.values[np.triu_indices_from(rho.values, 1)].mean(), 3))

## 5. Recursive Feature Elimination (RFECV, training split only)

Features are removed in rounds, a cross-validated random forest scores each size, and the **smallest subset within 1 standard error of the best score** is kept. Because the selection never sees the hold-out split, the hold-out AUC of the reduced model is an honest estimate.

In [ ]:
p = Xtr.shape[1]; step = max(1, int(RFE_STEP * p))
rf_base = RandomForestClassifier(150, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)
t0 = time.time()
rfe = RFECV(rf_base, step=step, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="roc_auc", min_features_to_select=5, n_jobs=1).fit(Xtr, ytr)
rfe_s = time.time() - t0
res = rfe.cv_results_
mean_, std_ = np.asarray(res["mean_test_score"]), np.asarray(res["std_test_score"])
sizes = list(res["n_features"]) if "n_features" in res else sorted({max(5, p - i * step) for i in range(len(mean_))})[: len(mean_)]
sizes = np.array(sizes)
rfe_tbl = pd.DataFrame({"n_features": sizes, "cv_auc_mean": mean_, "cv_auc_std": std_}).sort_values("n_features").reset_index(drop=True)
b = rfe_tbl["cv_auc_mean"].idxmax(); se = rfe_tbl.loc[b, "cv_auc_std"] / np.sqrt(5)
n_keep = int(rfe_tbl[rfe_tbl["cv_auc_mean"] >= rfe_tbl.loc[b, "cv_auc_mean"] - se]["n_features"].min())
print(f"RFECV took {rfe_s:.1f}s | best size {int(rfe_tbl.loc[b, 'n_features'])} (AUC {rfe_tbl.loc[b, 'cv_auc_mean']:.4f}) | chosen (1-SE rule): {n_keep} features")
plt.figure(figsize=(8, 4.5)); plt.errorbar(rfe_tbl["n_features"], rfe_tbl["cv_auc_mean"], yerr=rfe_tbl["cv_auc_std"], marker="o")
plt.axvline(n_keep, color="#c44e52", ls="--", label=f"chosen = {n_keep}"); plt.xlabel("number of features"); plt.ylabel("CV AUC (train split)"); plt.legend(); plt.title("RFECV"); plt.show()

In [ ]:
# RFECV chose the size; run a plain RFE down to exactly that size to get the surviving features and a full elimination order
from sklearn.feature_selection import RFE
rfe_final = RFE(RandomForestClassifier(150, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1), n_features_to_select=n_keep, step=step).fit(Xtr, ytr)
ranking = pd.Series(rfe_final.ranking_, index=Xtr.columns)                  # 1 = kept; larger = eliminated earlier
selected = list(Xtr.columns[rfe_final.support_])
print(f"Selected {len(selected)} of {p} features:"); print(selected)

## 6. Efficiency: full model vs RFE-selected vs SHAP top-k

In [ ]:
def evaluate(cols, label):
    m = RandomForestClassifier(300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)
    t0 = time.time(); m.fit(Xtr[cols], ytr); fit_s = time.time() - t0
    t0 = time.time(); pr = m.predict_proba(Xte[cols])[:, 1]; pred_s = time.time() - t0
    return {"feature_set": label, "n_features": len(cols), "holdout_auc": roc_auc_score(yte, pr), "fit_seconds": fit_s, "predict_seconds": pred_s,
            "model_size_kb": len(pickle.dumps(m)) / 1024}
shap_top = list(imp["shap_RandomForest"].sort_values(ascending=False).index[:n_keep])
eff = pd.DataFrame([evaluate(list(X.columns), "all features"), evaluate(selected, "RFE selected"), evaluate(shap_top, f"SHAP top-{n_keep}"),
                    evaluate(list(cons.index[:n_keep]), f"consensus top-{n_keep}")])
eff["auc_change_vs_all"] = eff["holdout_auc"] - eff.loc[0, "holdout_auc"]; eff["speedup_fit"] = eff.loc[0, "fit_seconds"] / eff["fit_seconds"]
eff["size_reduction"] = 1 - eff["model_size_kb"] / eff.loc[0, "model_size_kb"]
eff.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=eff, x="feature_set", y="holdout_auc", ax=ax[0], color="#4c72b0"); ax[0].set_ylim(eff["holdout_auc"].min() - .02, eff["holdout_auc"].max() + .02); ax[0].set_title("Hold-out AUC")
sns.barplot(data=eff, x="feature_set", y="fit_seconds", ax=ax[1], color="#c44e52"); ax[1].set_title("Fit time (s)")
sns.barplot(data=eff, x="feature_set", y="model_size_kb", ax=ax[2], color="#55a868"); ax[2].set_title("Model size (KB)")
for a in ax: a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 7. Save outputs

In [ ]:
out_cols = ([id_col] if id_col else []) + [c for c in selected]
sel_df = pd.concat([df[[id_col]] if id_col else pd.DataFrame(index=df.index), y.rename(target_name), X[selected]], axis=1)
out_path = PROC / "womens_clothing_feature_selected.csv"; sel_df.to_csv(out_path, index=False)

full = pd.concat([imp.add_prefix("imp_"), ranks.add_prefix("rank_"), cons[["mean_rank", "rank_std", "n_methods_in_top"]]], axis=1)
full["rfe_ranking"] = ranking.reindex(full.index); full["rfe_selected"] = full.index.isin(selected)
full.sort_values("mean_rank").round(6).reset_index(names="feature").to_csv(REPORTS / "task_24_feature_importance_comparison.csv", index=False)
rfe_tbl.assign(chosen=(rfe_tbl["n_features"] == n_keep)).round(6).to_csv(REPORTS / "task_24_rfe_results.csv", index=False)
eff.round(6).to_csv(REPORTS / "task_24_model_efficiency.csv", index=False)
status = pd.DataFrame([{"task": "Task 24 - Feature importance, SHAP, RFE", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(sel_df), "target": target_name, "candidate_features": X.shape[1], "excluded_columns": len(drop),
    "rfe_selected_features": len(selected), "holdout_auc_all": round(eff.loc[0, "holdout_auc"], 4), "holdout_auc_rfe": round(eff.loc[1, "holdout_auc"], 4),
    "fit_speedup_rfe": round(eff.loc[1, "speedup_fit"], 2), "mean_pairwise_rank_corr": round(rho.values[np.triu_indices_from(rho.values, 1)].mean(), 3),
    "top_feature_consensus": cons.index[0], "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_24_analysis_status.csv", index=False); status.T

## 8. Key takeaways (fill in after running)

- Which features rank in the top 5 across *all* methods (`n_methods_in_top`)? Features that only one method likes are less trustworthy.
- How correlated are impurity, permutation and SHAP rankings? Impurity importance favours high-cardinality / continuous features; permutation and SHAP are fairer.
- How many features did RFE remove, and what happened to hold-out AUC, fit time and model size?
- Importance is not causation, and with strong leakage controls AUCs are usually modest; compare against the full-feature model, not against 1.0.
- `womens_clothing_feature_selected.csv` is the compact model-ready dataset for later tasks (selected on the training split only).